# Colab: query CSV 임베딩 후 Pinecone 업로드

이 노트북은 Google Colab 기준으로 작성되었습니다.

## Colab Secrets 필요 값

- `OPENAI_API_KEY`
- `PINECONE_API_KEY`
- `PINECONE_INDEX`

## 처리 대상

- `qualified_query.csv` → `qualify_conditions` namespace
- `preffered_query.csv` → `preffered_conditions` namespace

각 CSV의 `hire_query` 컬럼을 `text-embedding-3-small`로 임베딩하고, `condition` 컬럼은 Pinecone metadata의 `condition` key에 저장합니다.


In [1]:
!pip -q install openai pinecone pandas tqdm


In [2]:
import hashlib
import math
import time
from typing import Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm

from google.colab import userdata
from openai import OpenAI
from pinecone import Pinecone


OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
PINECONE_INDEX = userdata.get("PINECONE_INDEX")

missing_keys = [
    key_name
    for key_name, key_value in {
        "OPENAI_API_KEY": OPENAI_API_KEY,
        "PINECONE_API_KEY": PINECONE_API_KEY,
        "PINECONE_INDEX": PINECONE_INDEX,
    }.items()
    if not key_value
]

if missing_keys:
    raise RuntimeError(
        "Colab Secrets에 필요한 값이 없습니다. "
        f"누락된 키: {missing_keys}"
    )

EMBEDDING_MODEL = "text-embedding-3-small"

print("Secrets 로드 완료")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Pinecone index: {PINECONE_INDEX}")


Secrets 로드 완료
Embedding model: text-embedding-3-small
Pinecone index: humour-manual


In [3]:
def get_embedding_client() -> OpenAI:
    """OpenAI embedding client를 반환합니다."""
    return OpenAI(api_key=OPENAI_API_KEY)


def get_pinecone_index():
    """Pinecone index 객체를 반환합니다."""
    pc = Pinecone(api_key=PINECONE_API_KEY)
    return pc.Index(PINECONE_INDEX)


embedding_client = get_embedding_client()
pinecone_index = get_pinecone_index()

print("OpenAI / Pinecone client 생성 완료")


OpenAI / Pinecone client 생성 완료


In [4]:
REQUIRED_COLUMNS = ["condition", "hire_query"]


def read_csv_with_fallback(path: str) -> pd.DataFrame:
    """CSV 인코딩을 순차적으로 시도해서 읽습니다."""
    encodings = ["utf-8-sig", "utf-8", "cp949"]
    last_error = None

    for encoding in encodings:
        try:
            df = pd.read_csv(path, encoding=encoding)
            print(f"[로드 성공] {path} / encoding={encoding} / rows={len(df):,}")
            return df
        except UnicodeDecodeError as e:
            last_error = e
        except FileNotFoundError:
            raise FileNotFoundError(
                f"파일을 찾을 수 없습니다: {path}\n"
                "Colab 왼쪽 파일 영역에 CSV를 업로드했는지 확인하세요."
            )

    raise RuntimeError(
        f"CSV 인코딩을 확인해주세요. 시도한 인코딩: {encodings}. 마지막 오류: {last_error}"
    )


def validate_required_columns(df: pd.DataFrame, path: str) -> None:
    """condition, hire_query 컬럼이 있는지 확인합니다."""
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]

    if missing:
        raise ValueError(
            f"{path}에 필수 컬럼이 없습니다. "
            f"누락 컬럼: {missing} / 현재 컬럼: {list(df.columns)}"
        )


def load_query_csv(path: str, max_rows: Optional[int] = None) -> pd.DataFrame:
    """CSV를 읽고 필요한 컬럼만 정리합니다."""
    df = read_csv_with_fallback(path)
    validate_required_columns(df, path)

    df = df[REQUIRED_COLUMNS].copy()
    df = df.fillna("")

    before = len(df)
    df = df[df["hire_query"].astype(str).str.strip() != ""].copy()
    after = len(df)

    if before != after:
        print(f"[빈 hire_query 제외] {before - after:,}행 제거")

    if max_rows is not None:
        df = df.head(max_rows).copy()
        print(f"[샘플 모드] {path}: 앞 {max_rows}행만 처리")

    print(f"[처리 대상] {path}: {len(df):,}행")
    return df


In [5]:
def get_embedding(query: str, max_retries: int = 3, sleep_seconds: float = 1.5) -> List[float]:
    """hire_query 텍스트를 text-embedding-3-small로 임베딩합니다."""
    query = str(query).strip()

    if not query:
        raise ValueError("임베딩할 query가 비어 있습니다.")

    for attempt in range(1, max_retries + 1):
        try:
            response = get_embedding_client().embeddings.create(
                model=EMBEDDING_MODEL,
                input=query,
            )
            return response.data[0].embedding

        except Exception as e:
            print(f"[Embedding 오류] attempt={attempt}/{max_retries} / error={e}")

            if attempt == max_retries:
                raise

            time.sleep(sleep_seconds * attempt)


In [6]:
def make_vector_id(prefix: str, row_index: int, hire_query: str, condition: str) -> str:
    """동일 데이터 재실행 시 같은 ID가 나오도록 안정적인 vector id를 생성합니다."""
    raw = f"{prefix}|{row_index}|{hire_query}|{condition}"
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]
    return f"{prefix}-{row_index}-{digest}"


def chunk_list(items: List[Dict], batch_size: int) -> List[List[Dict]]:
    """리스트를 batch 단위로 나눕니다."""
    return [items[i:i + batch_size] for i in range(0, len(items), batch_size)]


def build_vectors_from_df(df: pd.DataFrame, id_prefix: str) -> List[Dict]:
    """DataFrame의 hire_query를 임베딩하고 Pinecone upsert vector list를 생성합니다."""
    vectors = []

    for row_index, row in tqdm(
        df.reset_index(drop=True).iterrows(),
        total=len(df),
        desc=f"Embedding {id_prefix}",
    ):
        hire_query = str(row["hire_query"]).strip()
        condition = str(row["condition"]).strip()

        embedding = get_embedding(hire_query)

        vectors.append({
            "id": make_vector_id(
                prefix=id_prefix,
                row_index=row_index,
                hire_query=hire_query,
                condition=condition,
            ),
            "values": embedding,
            "metadata": {
                "condition": condition
            },
        })

    return vectors


def upsert_vectors_to_pinecone(vectors: List[Dict], namespace: str, batch_size: int = 100) -> None:
    """생성된 vector를 Pinecone namespace에 batch upsert합니다."""
    if not vectors:
        print(f"[스킵] namespace={namespace}: 업로드할 vector가 없습니다.")
        return

    total_batches = math.ceil(len(vectors) / batch_size)

    print(f"[Upsert 시작] namespace={namespace} / vectors={len(vectors):,} / batch_size={batch_size}")

    for batch in tqdm(
        chunk_list(vectors, batch_size),
        total=total_batches,
        desc=f"Upserting {namespace}",
    ):
        pinecone_index.upsert(
            vectors=batch,
            namespace=namespace,
        )

    print(f"[Upsert 완료] namespace={namespace} / vectors={len(vectors):,}")


In [7]:
def process_query_csv_to_pinecone(
    csv_path: str,
    namespace: str,
    id_prefix: str,
    max_rows: Optional[int] = None,
    batch_size: int = 100,
) -> List[Dict]:
    """CSV 파일을 읽어 hire_query 임베딩 후 Pinecone namespace에 저장합니다."""
    print("=" * 90)
    print(f"[처리 시작] csv={csv_path} → namespace={namespace}")

    df = load_query_csv(csv_path, max_rows=max_rows)

    print(f"[임베딩 예정 행 수] {len(df):,}")
    print("주의: 각 행마다 OpenAI Embedding API 호출이 발생합니다.")

    vectors = build_vectors_from_df(
        df=df,
        id_prefix=id_prefix,
    )

    upsert_vectors_to_pinecone(
        vectors=vectors,
        namespace=namespace,
        batch_size=batch_size,
    )

    print(f"[처리 완료] csv={csv_path} → namespace={namespace}")
    print("=" * 90)

    return vectors


In [8]:
# None이면 전체 행 처리
# 테스트로 앞 2행만 처리하려면 MAX_ROWS = 2로 변경하세요.
MAX_ROWS = None

# Pinecone upsert batch size
BATCH_SIZE = 100


In [9]:
qualified_vectors = process_query_csv_to_pinecone(
    csv_path="qualified_query.csv",
    namespace="qualify_conditions",
    id_prefix="qualified",
    max_rows=MAX_ROWS,
    batch_size=BATCH_SIZE,
)


[처리 시작] csv=qualified_query.csv → namespace=qualify_conditions
[로드 성공] qualified_query.csv / encoding=utf-8-sig / rows=658
[처리 대상] qualified_query.csv: 658행
[임베딩 예정 행 수] 658
주의: 각 행마다 OpenAI Embedding API 호출이 발생합니다.


Embedding qualified:   0%|          | 0/658 [00:00<?, ?it/s]

[Upsert 시작] namespace=qualify_conditions / vectors=658 / batch_size=100


Upserting qualify_conditions:   0%|          | 0/7 [00:00<?, ?it/s]

[Upsert 완료] namespace=qualify_conditions / vectors=658
[처리 완료] csv=qualified_query.csv → namespace=qualify_conditions


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
preffered_vectors = process_query_csv_to_pinecone(
    csv_path="preffered_query.csv",
    namespace="preffered_conditions",
    id_prefix="preffered",
    max_rows=MAX_ROWS,
    batch_size=BATCH_SIZE,
)


[처리 시작] csv=preffered_query.csv → namespace=preffered_conditions
[로드 성공] preffered_query.csv / encoding=utf-8-sig / rows=389
[처리 대상] preffered_query.csv: 389행
[임베딩 예정 행 수] 389
주의: 각 행마다 OpenAI Embedding API 호출이 발생합니다.


Embedding preffered:   0%|          | 0/389 [00:00<?, ?it/s]

[Upsert 시작] namespace=preffered_conditions / vectors=389 / batch_size=100


Upserting preffered_conditions:   0%|          | 0/4 [00:00<?, ?it/s]

[Upsert 완료] namespace=preffered_conditions / vectors=389
[처리 완료] csv=preffered_query.csv → namespace=preffered_conditions


In [12]:
# namespace별 vector count가 바로 반영되지 않을 수 있으니,
# 필요하면 몇 초 후 이 셀을 다시 실행하세요.

stats = pinecone_index.describe_index_stats()

print("Pinecone index stats:")
print(stats)


Pinecone index stats:
DescribeIndexStatsResponse(dimension=1536, total_vector_count=1094, metric='cosine', namespaces=3)


In [13]:
def query_namespace(text: str, namespace: str, top_k: int = 5):
    """입력 문장을 임베딩한 뒤 지정 namespace에서 유사 condition을 조회합니다."""
    embedding = get_embedding(text)

    result = pinecone_index.query(
        vector=embedding,
        namespace=namespace,
        top_k=top_k,
        include_metadata=True,
    )

    return result


# 예시:
# query_namespace(
#     text="Python과 FastAPI 기반 백엔드 개발자를 찾고 있음",
#     namespace="qualify_conditions",
#     top_k=5,
# )


## 메모

- `qualified_query.csv`는 `qualify_conditions` namespace에 저장됩니다.
- `preffered_query.csv`는 `preffered_conditions` namespace에 저장됩니다.
- metadata에는 요청대로 `condition` key만 저장합니다.
- 같은 CSV를 다시 실행하면 동일 row 기준으로 동일한 vector ID가 생성되므로 기존 vector를 overwrite합니다.
- 테스트만 할 경우 `MAX_ROWS = 2`로 바꾼 뒤 실행하세요.
